# 02 — Caracterização do dataset

**Entregável E3 / Bloco B2** (IBM8924 — AC de Projeto, Grupo 1).

Lê o dataset gerado por `01_dados.ipynb` (`data/processed/s2_patches.npz`) e produz:
distribuição de classes, estatísticas descritivas por banda, histograma do índice
espectral (NDVI/GNDVI), grade de exemplos por classe, e comentário sobre
desbalanceamento/artefatos.

Rode `01_dados.ipynb` até o fim antes deste notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataset_utils import (
    BAND_NAMES, load_dataset, compute_ndvi_gndvi, rgb_composite,
)

CLASS_NAMES = {0: 'Baixo vigor', 1: 'Médio vigor', 2: 'Alto vigor'}

data = load_dataset('../data/processed/s2_patches.npz')
X, y, split = data['X'], data['y'], data['split']

print('X:', X.shape, '| y:', y.shape)
print('Partições:', dict(zip(*np.unique(split, return_counts=True))))


## 1. Distribuição de classes (geral e por partição)

In [ ]:
df_meta = pd.DataFrame({'label': y, 'split': split})
df_meta['classe'] = df_meta['label'].map(CLASS_NAMES)

counts = df_meta.groupby(['split', 'classe']).size().unstack(fill_value=0)
counts = counts.reindex(['train', 'val', 'test'])
print(counts)

counts.plot(kind='bar', figsize=(7, 4))
plt.title('Distribuição de classes por partição')
plt.ylabel('Nº de recortes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('\nDistribuição geral:')
print(df_meta['classe'].value_counts())


## 2. Estatísticas por banda

In [ ]:
rows = []
for i, band in enumerate(BAND_NAMES):
    values = X[..., i].astype(np.float32)
    rows.append({
        'banda': band,
        'media': values.mean(),
        'desvio': values.std(),
        'minimo': values.min(),
        'maximo': values.max(),
    })

band_stats = pd.DataFrame(rows).set_index('banda')
band_stats


In [ ]:
# Mesmas estatísticas, quebradas por classe -- ajuda a ver se alguma banda já separa
# visualmente as classes de vigor antes mesmo do baseline.
rows = []
for cls, cls_name in CLASS_NAMES.items():
    mask = y == cls
    for i, band in enumerate(BAND_NAMES):
        values = X[mask, ..., i].astype(np.float32)
        rows.append({'classe': cls_name, 'banda': band, 'media': values.mean(), 'desvio': values.std()})

pd.DataFrame(rows).pivot(index='banda', columns='classe', values='media')


## 3. Índice espectral (NDVI / GNDVI)

NDVI = (NIR − RED) / (NIR + RED) (Rouse et al., 1974); GNDVI = (NIR − GREEN) / (NIR + GREEN)
(Gitelson et al., 1996); NIR=B8, RED=B4, GREEN=B3 (Sentinel-2).

In [ ]:
ndvi_means, gndvi_means = [], []
for patch in X:
    ndvi, gndvi = compute_ndvi_gndvi(patch)
    ndvi_means.append(ndvi.mean())
    gndvi_means.append(gndvi.mean())

ndvi_means = np.array(ndvi_means)
gndvi_means = np.array(gndvi_means)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for cls, cls_name in CLASS_NAMES.items():
    mask = y == cls
    axes[0].hist(ndvi_means[mask], bins=20, alpha=0.6, label=cls_name)
    axes[1].hist(gndvi_means[mask], bins=20, alpha=0.6, label=cls_name)

axes[0].set_title('NDVI médio por patch')
axes[0].set_xlabel('NDVI')
axes[0].legend()
axes[1].set_title('GNDVI médio por patch')
axes[1].set_xlabel('GNDVI')
axes[1].legend()
plt.tight_layout()
plt.show()

print('NDVI médio geral:', ndvi_means.mean(), '| desvio:', ndvi_means.std())


## 4. Grade de exemplos por classe (composição RGB)

In [ ]:
N_EXAMPLES = 5
rng = np.random.default_rng(42)

fig, axes = plt.subplots(len(CLASS_NAMES), N_EXAMPLES, figsize=(2.2 * N_EXAMPLES, 2.2 * len(CLASS_NAMES)))

for row, (cls, cls_name) in enumerate(CLASS_NAMES.items()):
    idx = np.where(y == cls)[0]
    chosen = rng.choice(idx, size=min(N_EXAMPLES, len(idx)), replace=False)
    for col in range(N_EXAMPLES):
        ax = axes[row, col]
        if col < len(chosen):
            ax.imshow(rgb_composite(X[chosen[col]]))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls_name)

plt.suptitle('Exemplos de recortes por classe de vigor')
plt.tight_layout()
plt.show()


## 5. Comentário sobre desbalanceamento e artefatos

*(preencher após rodar as células acima com os números/observações reais)*

- **Desbalanceamento:** o desenho de amostragem (`stratifiedSample`) força balanceamento
  por construção — descrever aqui qualquer desvio introduzido pela checagem de qualidade
  de `01_dados.ipynb` (patches descartados por excesso de pixels sem dado), que pode ter
  afetado uma classe mais do que outra.
- **Nuvem/sombra:** inspecionar visualmente a grade de exemplos da seção 4 — anotar se
  algum patch mostra nuvem residual, sombra de nuvem ou névoa não filtrada pelo limiar
  `CLOUDY_PIXEL_PERCENTAGE < 20`.
- **Sazonalidade:** os compostos usam sempre a janela jun–set (estação seca) do ano do
  ponto — comentar se isso é suficiente para reduzir variação sazonal entre anos
  diferentes (2019–2022) ou se ainda há efeito visível nas estatísticas da seção 2.